In [ ]:
"""
Gravitational Lensing Image Classification
============================================
Multi-class classification of strong lensing images into:
  - no: No substructure
  - sphere: Subhalo substructure
  - vort: Vortex substructure

Approach: Custom CNN in PyTorch
Evaluation: ROC curves + AUC scores (one-vs-rest)
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from itertools import cycle



In [ ]:
# 0. Configuration

DATA_DIR = Path("/Users/rishabhsharma/Downloads/dataset")
DEVICE = torch.device("cpu")
CLASS_NAMES = ["no", "sphere", "vort"]
CLASS_LABELS = {"no": 0, "sphere": 1, "vort": 2}
BATCH_SIZE = 64
NUM_EPOCHS = 15
LR = 1e-3
SEED = 42
OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")



In [ ]:
# 1a. Class distribution 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for idx, split in enumerate(["train", "val"]):
    counts = {cls: len(os.listdir(DATA_DIR / split / cls)) for cls in CLASS_NAMES}
    axes[idx].bar(counts.keys(), counts.values(), color=["#4C72B0", "#DD8452", "#55A868"])
    axes[idx].set_title(f"{split.capitalize()} Set Distribution")
    axes[idx].set_ylabel("Number of Images")
    for cls, count in counts.items():
        axes[idx].text(cls, count + 100, str(count), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: class_distribution.png")



In [ ]:
# 1b. Sample images from each class 
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for row, cls in enumerate(CLASS_NAMES):
    files = sorted(os.listdir(DATA_DIR / "train" / cls))[:5]
    for col, f in enumerate(files):
        img = np.load(DATA_DIR / "train" / cls / f)[0]
        axes[row, col].imshow(img, cmap="inferno")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=14, fontweight="bold", rotation=0, labelpad=50)
        if row == 0:
            axes[row, col].set_title(f"Sample {col + 1}")
fig.suptitle("Sample Images per Class", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_images.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: sample_images.png")



In [ ]:
# 1c. Pixel intensity statistics
print("\nPixel Intensity Statistics (sampled 500 per class):")
stats = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, cls in enumerate(CLASS_NAMES):
    files = sorted(os.listdir(DATA_DIR / "train" / cls))[:500]
    pixels = np.concatenate([np.load(DATA_DIR / "train" / cls / f).flatten() for f in files])
    stats[cls] = {"mean": pixels.mean(), "std": pixels.std(), "median": np.median(pixels)}
    print(f"  {cls:>8s}: mean={pixels.mean():.4f}, std={pixels.std():.4f}, "
          f"median={np.median(pixels):.4f}, max={pixels.max():.4f}")
    axes[idx].hist(pixels[pixels > 0.01], bins=100, color=["#4C72B0", "#DD8452", "#55A868"][idx], alpha=0.8)
    axes[idx].set_title(f"{cls} (non-zero pixels)")
    axes[idx].set_xlabel("Pixel Value")
    axes[idx].set_ylabel("Count")
    axes[idx].set_yscale("log")
fig.suptitle("Pixel Intensity Distributions (log scale)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pixel_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: pixel_distributions.png")



In [ ]:
# 1d. Mean image per class 
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for idx, cls in enumerate(CLASS_NAMES):
    files = sorted(os.listdir(DATA_DIR / "train" / cls))[:1000]
    mean_img = np.mean([np.load(DATA_DIR / "train" / cls / f)[0] for f in files], axis=0)
    im = axes[idx].imshow(mean_img, cmap="inferno")
    axes[idx].set_title(f"Mean: {cls}", fontsize=13, fontweight="bold")
    axes[idx].axis("off")
    plt.colorbar(im, ax=axes[idx], fraction=0.046)
fig.suptitle("Average Image per Class (1000 samples)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mean_images.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: mean_images.png")



In [ ]:
# 1e. Radial intensity profile
fig, ax = plt.subplots(figsize=(8, 5))
center = 75  # 150/2
y, x = np.ogrid[:150, :150]
r = np.sqrt((x - center) ** 2 + (y - center) ** 2).astype(int)
for cls, color in zip(CLASS_NAMES, ["#4C72B0", "#DD8452", "#55A868"]):
    files = sorted(os.listdir(DATA_DIR / "train" / cls))[:200]
    mean_img = np.mean([np.load(DATA_DIR / "train" / cls / f)[0] for f in files], axis=0)
    radial_profile = np.array([mean_img[r == ri].mean() for ri in range(0, 75)])
    ax.plot(radial_profile, label=cls, color=color, linewidth=2)
ax.set_xlabel("Distance from Center (pixels)")
ax.set_ylabel("Mean Intensity")
ax.set_title("Radial Intensity Profile per Class")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "radial_profiles.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: radial_profiles.png")



In [ ]:
# 1f. Difference images (class vs class) 
mean_imgs = {}
for cls in CLASS_NAMES:
    files = sorted(os.listdir(DATA_DIR / "train" / cls))[:1000]
    mean_imgs[cls] = np.mean([np.load(DATA_DIR / "train" / cls / f)[0] for f in files], axis=0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [("sphere", "no"), ("vort", "no"), ("vort", "sphere")]
for idx, (a, b) in enumerate(pairs):
    diff = mean_imgs[a] - mean_imgs[b]
    im = axes[idx].imshow(diff, cmap="RdBu_r", vmin=-0.02, vmax=0.02)
    axes[idx].set_title(f"{a} - {b}", fontsize=13, fontweight="bold")
    axes[idx].axis("off")
    plt.colorbar(im, ax=axes[idx], fraction=0.046)
fig.suptitle("Difference Maps Between Class Means", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "difference_maps.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: difference_maps.png")

print("\nImage Analysis Summary:")
print("  - All images are 1x150x150, min-max normalized to [0, 1]")
print("  - Images are sparse (mostly dark background, bright central lensing features)")
print("  - Classes are perfectly balanced (10k train, 2.5k val each)")
print("  - Differences are subtle — concentrated around the Einstein ring region")
print("  - Substructure manifests as perturbations in the ring/arc morphology")




In [ ]:
# PART 2: DATASET & DATALOADER
class LensingDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        self.samples = []
        self.augment = augment
        for cls_name, label in CLASS_LABELS.items():
            cls_dir = Path(root_dir) / cls_name
            for f in os.listdir(cls_dir):
                if f.endswith(".npy"):
                    self.samples.append((cls_dir / f, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.load(path).astype(np.float32)  # (1, 150, 150)
        img = torch.from_numpy(img)

        if self.augment:
            # Random horizontal flip
            if torch.rand(1).item() > 0.5:
                img = torch.flip(img, [2])
            # Random vertical flip
            if torch.rand(1).item() > 0.5:
                img = torch.flip(img, [1])
            # Random 90-degree rotations
            k = torch.randint(0, 4, (1,)).item()
            if k > 0:
                img = torch.rot90(img, k, [1, 2])

        return img, label


In [ ]:


train_dataset = LensingDataset(DATA_DIR / "train", augment=True)
val_dataset = LensingDataset(DATA_DIR / "val", augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")




In [ ]:
# PART 3: MODEL ARCHITECTURE
class LensingCNN(nn.Module):
    """
    Custom CNN for gravitational lensing classification.

    Architecture rationale:
    - Single-channel input (not RGB) → custom first conv layer
    - 4 conv blocks with increasing filters (32→64→128→256)
    - BatchNorm + ReLU + MaxPool in each block
    - Global Average Pooling → compact feature vector
    - Dropout for regularization
    - 3-class softmax output
    """

    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 1x150x150 → 32x75x75
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 32x75x75 → 64x37x37
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 64x37x37 → 128x18x18
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4: 128x18x18 → 256x9x9
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),     # 256x9x9 → 256x1x1
            nn.Flatten(),                # → 256
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:


model = LensingCNN(num_classes=3).to(DEVICE)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)




In [ ]:
# PART 4: TRAINING

print("\n" + "=" * 60)
print("PART 3: TRAINING")
print("=" * 60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):


In [ ]:
    # Train 
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total



In [ ]:
    # Validate 
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / total
    val_acc = 100.0 * correct / total

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%  |  "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pth")

print(f"\nBest Validation Accuracy: {best_val_acc:.2f}%")



In [ ]:
# Training curves 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train", linewidth=2)
ax1.plot(history["val_loss"], label="Validation", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], label="Train", linewidth=2)
ax2.plot(history["val_acc"], label="Validation", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Training & Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: training_curves.png")




In [ ]:
# PART 5: EVALUATION — ROC CURVES & AUC

# Load best model
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pth", map_location=DEVICE))
model.eval()

all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_labels = np.concatenate(all_labels)
all_probs = np.concatenate(all_probs)
all_preds = np.argmax(all_probs, axis=1)



In [ ]:
# Classification Report 
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))



In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=ax, cbar=True)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: confusion_matrix.png")



In [ ]:
# ROC Curves (One-vs-Rest)
# Binarize labels for OvR
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(all_labels, classes=[0, 1, 2])

fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#4C72B0", "#DD8452", "#55A868"]
auc_scores = {}

for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls] = roc_auc
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{cls} (AUC = {roc_auc:.4f})")

# Macro average
all_fpr = np.unique(np.concatenate([roc_curve(y_bin[:, i], all_probs[:, i])[0] for i in range(3)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    mean_tpr += np.interp(all_fpr, fpr, tpr)
mean_tpr /= 3
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, color="black", linewidth=2.5, linestyle="--",
        label=f"Macro Average (AUC = {macro_auc:.4f})")

ax.plot([0, 1], [0, 1], "k:", alpha=0.4, label="Random (AUC = 0.5)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves (One-vs-Rest)", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()

print("\nAUC Scores (One-vs-Rest):")
for cls, score in auc_scores.items():
    print(f"  {cls:>8s}: {score:.4f}")
print(f"  {'Macro':>8s}: {macro_auc:.4f}")
print("\nSaved: roc_curves.png")



In [ ]:
# Final Summary
print(f"  Model: Custom 4-block CNN (PyTorch)")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Best Val Accuracy: {best_val_acc:.2f}%")
print(f"  Macro AUC: {macro_auc:.4f}")
print(f"  All results saved to: {OUTPUT_DIR}")



# train_and_eval 

In [ ]:
"""
Fast training & evaluation script.
Pre-loads all .npy files into memory for speed.
EDA already done — this just trains and evaluates.
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize

DATA_DIR = Path("/Users/rishabhsharma/Downloads/dataset")
OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)
DEVICE = torch.device("cpu")
CLASS_NAMES = ["no", "sphere", "vort"]
BATCH_SIZE = 128
NUM_EPOCHS = 15
LR = 1e-3
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)



In [ ]:
# Load all data into memory
def load_split(split):
    images, labels = [], []
    for label, cls in enumerate(CLASS_NAMES):
        cls_dir = DATA_DIR / split / cls
        files = [f for f in os.listdir(cls_dir) if f.endswith(".npy")]
        for f in files:
            img = np.load(cls_dir / f, allow_pickle=False).astype(np.float32)
            images.append(img)
            labels.append(label)
    images = np.stack(images)
    labels = np.array(labels, dtype=np.int64)
    return images, labels


In [ ]:

print("Loading training data...")
X_train, y_train = load_split("train")
print(f"  Train: {X_train.shape}, labels: {y_train.shape}")

print("Loading validation data...")
X_val, y_val = load_split("val")
print(f"  Val: {X_val.shape}, labels: {y_val.shape}")

# Create tensor datasets
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)




In [ ]:
# Model
class LensingCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(256, 128), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:


model = LensingCNN().to(DEVICE)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)



In [ ]:
# Training

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

print(f"\nTraining for {NUM_EPOCHS} epochs on {DEVICE}...")
for epoch in range(NUM_EPOCHS):
    model.train()
    rloss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        # Simple augmentation: random flips
        if torch.rand(1).item() > 0.5:
            imgs = torch.flip(imgs, [3])
        if torch.rand(1).item() > 0.5:
            imgs = torch.flip(imgs, [2])

        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        rloss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)

    train_loss = rloss / total
    train_acc = 100.0 * correct / total

    model.eval()
    rloss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            loss = criterion(out, lbls)
            rloss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == lbls).sum().item()
            total += imgs.size(0)

    val_loss = rloss / total
    val_acc = 100.0 * correct / total
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    lr = optimizer.param_groups[0]["lr"]
    print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS}  "
          f"Train: {train_loss:.4f} / {train_acc:.1f}%  "
          f"Val: {val_loss:.4f} / {val_acc:.1f}%  LR: {lr:.1e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model.pth")

print(f"\nBest Val Accuracy: {best_val_acc:.2f}%")



In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train", linewidth=2)
ax1.plot(history["val_loss"], label="Val", linewidth=2)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history["train_acc"], label="Train", linewidth=2)
ax2.plot(history["val_acc"], label="Val", linewidth=2)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: training_curves.png")




In [ ]:
# Evaluation: ROC + AUC

model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pth", map_location=DEVICE))
model.eval()

all_probs, all_labels_list = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        out = model(imgs.to(DEVICE))
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels_list.append(lbls.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels_list)
all_preds = np.argmax(all_probs, axis=1)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))



In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: confusion_matrix.png")



In [ ]:
# ROC Curves
y_bin = label_binarize(all_labels, classes=[0, 1, 2])
fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#4C72B0", "#DD8452", "#55A868"]
auc_scores = {}

for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls] = roc_auc
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f"{cls} (AUC = {roc_auc:.4f})")

# Macro average ROC
all_fpr = np.unique(np.concatenate([roc_curve(y_bin[:, i], all_probs[:, i])[0] for i in range(3)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    mean_tpr += np.interp(all_fpr, fpr, tpr)
mean_tpr /= 3
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "k--", linewidth=2.5, label=f"Macro Avg (AUC = {macro_auc:.4f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4, label="Random (AUC = 0.5)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (One-vs-Rest)")
ax.legend(loc="lower right", fontsize=11); ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()

print("\nAUC Scores:")
for cls, s in auc_scores.items():
    print(f"  {cls:>8s}: {s:.4f}")
print(f"  {'Macro':>8s}: {macro_auc:.4f}")
print("\nSaved: roc_curves.png")
print(f"\nAll outputs in: {OUTPUT_DIR}")


In [ ]:
"""
Extended training: continue from best_model.pth for 35 more epochs
with cosine annealing scheduler for better convergence.
"""

import os, sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize

DATA_DIR = Path("/Users/rishabhsharma/Downloads/dataset")
OUTPUT_DIR = DATA_DIR / "results"
DEVICE = torch.device("cpu")
CLASS_NAMES = ["no", "sphere", "vort"]
BATCH_SIZE = 128
NUM_EPOCHS = 35
LR = 3e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)




In [ ]:
# Load data
def load_split(split):
    images, labels = [], []
    for label, cls in enumerate(CLASS_NAMES):
        cls_dir = DATA_DIR / split / cls
        for f in os.listdir(cls_dir):
            if f.endswith(".npy"):
                images.append(np.load(cls_dir / f, allow_pickle=False).astype(np.float32))
                labels.append(label)
    return np.stack(images), np.array(labels, dtype=np.int64)


In [ ]:

print("Loading data...", flush=True)
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
print(f"Train: {X_train.shape}, Val: {X_val.shape}", flush=True)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)),
                        batch_size=BATCH_SIZE, shuffle=False)




In [ ]:
class LensingCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(256, 128), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:


model = LensingCNN().to(DEVICE)
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pth", map_location=DEVICE))
print(f"Loaded best_model.pth, continuing training...", flush=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    rloss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        if torch.rand(1).item() > 0.5:
            imgs = torch.flip(imgs, [3])
        if torch.rand(1).item() > 0.5:
            imgs = torch.flip(imgs, [2])

        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        rloss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)

    train_loss, train_acc = rloss / total, 100.0 * correct / total

    model.eval()
    rloss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            loss = criterion(out, lbls)
            rloss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == lbls).sum().item()
            total += imgs.size(0)

    val_loss, val_acc = rloss / total, 100.0 * correct / total
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    lr = optimizer.param_groups[0]["lr"]
    print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS}  "
          f"Train: {train_loss:.4f} / {train_acc:.1f}%  "
          f"Val: {val_loss:.4f} / {val_acc:.1f}%  LR: {lr:.1e}", flush=True)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), OUTPUT_DIR / "best_model_v2.pth")

print(f"\nBest Val Accuracy: {best_val_acc:.2f}%", flush=True)



In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history["train_loss"], label="Train", linewidth=2)
ax1.plot(history["val_loss"], label="Val", linewidth=2)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss (Extended Training)")
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history["train_acc"], label="Train", linewidth=2)
ax2.plot(history["val_acc"], label="Val", linewidth=2)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.set_title("Accuracy (Extended Training)")
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: training_curves.png", flush=True)

# Evaluation
print("\n=== EVALUATION ===", flush=True)
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model_v2.pth", map_location=DEVICE))
model.eval()

all_probs, all_labels_list = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        out = model(imgs.to(DEVICE))
        all_probs.append(torch.softmax(out, dim=1).cpu().numpy())
        all_labels_list.append(lbls.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels_list)
all_preds = np.argmax(all_probs, axis=1)

print("\nClassification Report:", flush=True)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))



In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: confusion_matrix.png", flush=True)



In [ ]:
# ROC Curves
y_bin = label_binarize(all_labels, classes=[0, 1, 2])
fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#4C72B0", "#DD8452", "#55A868"]
auc_scores = {}

for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls] = roc_auc
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f"{cls} (AUC = {roc_auc:.4f})")

all_fpr = np.unique(np.concatenate([roc_curve(y_bin[:, i], all_probs[:, i])[0] for i in range(3)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    mean_tpr += np.interp(all_fpr, fpr, tpr)
mean_tpr /= 3
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, "k--", linewidth=2.5, label=f"Macro Avg (AUC = {macro_auc:.4f})")
ax.plot([0, 1], [0, 1], "k:", alpha=0.4, label="Random (AUC = 0.5)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (One-vs-Rest)")
ax.legend(loc="lower right", fontsize=11); ax.grid(alpha=0.3)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()

print("\nAUC Scores:", flush=True)
for cls, s in auc_scores.items():
    print(f"  {cls:>8s}: {s:.4f}")
print(f"  {'Macro':>8s}: {macro_auc:.4f}")
print("\nAll outputs saved to:", OUTPUT_DIR, flush=True)
